In [7]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [8]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import numpy as np

from clonalg.antibody.real_antibody import RealAntibody, RealAntibodyBuilder
from clonalg.model.clonalg_optimization import OptimizationClonalg
from clonalg.problems.problem import *
from clonalg.visualization.visualization import *

In [10]:
griewank = Griewank()
n_dims = 2
bounds = [(-18.00, 18.00)] * n_dims


def factory() -> RealAntibody:
    genes = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
    return (
        RealAntibodyBuilder()
        .with_genes(genes)
        .with_bounds(bounds)
        .with_cost_fn(griewank)
        .build()
    )


clonalg = OptimizationClonalg(
    population_size=30,
    clone_factor=0.2,
    n_generations=1000,
    antibody_factory=factory,
    hypermutation_strategy="affinity",
    rho=3.0,
    suppression_threshold=5.0,
)

memory = clonalg.run()
memory.sort(key=lambda ab: ab.affinity(None), reverse=True)

100%|██████████| 1000/1000 [00:11<00:00, 88.23it/s]


In [11]:
best = memory[0]
paths = list(map(lambda x: np.array(x.genes).reshape(1, 2), memory))

print(f"Best solution: x = {best.genes}")
print(f"f(x)         = {griewank(best.genes):.6f}")
print(f"affinity     = {best.affinity(None):.6f}")

Best solution: x = [ 0.0036581  -0.02980684]
f(x)         = 0.000229
affinity     = 0.999771


In [12]:
plot_3d_surface_without_grid(griewank, bounds=(-20.00, 20.00), grid_size=100)
plot_contour_and_paths(griewank, paths, bounds=(-20.00, 20.00))